# MeCrab Python API - Advanced Features

**Date:** 2026-01-02  
**Author:** COOLJAPAN OU (Team KitaSan)  
**License:** MIT OR Apache-2.0

This notebook demonstrates advanced features:
- IPA (International Phonetic Alphabet) pronunciation
- Word embeddings (Word2Vec)
- Semantic similarity
- Pythonic dictionary API

## Setup

**Prerequisites:**
1. MeCrab Python package installed
2. IPADIC dictionary installed
3. *(Optional)* Word vectors - either download or train (see section 4)

In [ ]:
import mecrab
print(f"MeCrab version: {mecrab.version()}")

## 1. IPA Pronunciation - One-Shot Conversion

Convert Japanese text directly to IPA (International Phonetic Alphabet) pronunciation.

This is the **key feature** - instant phonetic conversion!

In [ ]:
# Initialize with IPA support
m_ipa = mecrab.MeCrab(with_ipa=True)

# One-shot IPA conversion
text = "東京に行く"
ipa_list = m_ipa.to_ipa(text)

print(f"Original: {text}")
print(f"IPA (list): {ipa_list}")
print(f"IPA (joined): {' '.join(ipa_list)}")

In [ ]:
# Even simpler: get as a single string
ipa_text = m_ipa.to_ipa_text("東京に行く")
print(f"IPA: /{ipa_text}/")

# With custom separator
ipa_hyphenated = m_ipa.to_ipa_text("東京に行く", separator="-")
print(f"IPA (hyphenated): {ipa_hyphenated}")

### IPA Examples - Common Phrases

In [ ]:
phrases = [
    "こんにちは",
    "ありがとうございます",
    "おはようございます",
    "さようなら",
    "すみません"
]

print("Japanese → IPA Pronunciation\n" + "="*50)
for phrase in phrases:
    ipa = m_ipa.to_ipa_text(phrase)
    print(f"{phrase:20s} → /{ipa}/")

## 2. Pythonic Dictionary API

Get structured output with all morpheme information as Python dicts.

In [ ]:
# Parse to dictionary (Pythonic API)
text = "東京に行く"
morphemes = m_ipa.parse_to_dict(text)

print(f"Parsed: {text}\n")
for morph in morphemes:
    print(f"Surface: {morph['surface']}")
    print(f"  POS: {morph['pos']}")
    print(f"  Reading: {morph.get('reading', 'N/A')}")
    print(f"  IPA: /{morph.get('ipa', 'N/A')}/")
    print()

### Extract Specific Fields

In [ ]:
text = "私は東京で美味しいラーメンを食べました"
morphemes = m_ipa.parse_to_dict(text)

# Extract nouns with readings and IPA
nouns = [(m['surface'], m.get('reading'), m.get('ipa')) 
         for m in morphemes if m['pos'] == '名詞']

print("Nouns in sentence:\n")
print(f"{'Surface':<15} {'Reading':<15} {'IPA':<20}")
print("="*50)
for surface, reading, ipa in nouns:
    print(f"{surface:<15} {reading or 'N/A':<15} /{ipa or 'N/A'}/")

## 3. Word Embeddings - Cosine Similarity

**Note:** This requires word vectors. See section 4 below for download/training instructions.

### Quick Setup - Download or Check for Vectors

In [ ]:
import os

vector_path = "vectors.bin"

if not os.path.exists(vector_path):
    print("Downloading pre-trained vectors from HuggingFace...")
    try:
        from huggingface_hub import hf_hub_download
        vector_path = hf_hub_download(
            repo_id="KitaSan/mecrab-jawiki-word2vec",
            filename="vectors.bin",
            repo_type="dataset"
        )
        print(f"✓ Downloaded to: {vector_path}")
    except ImportError:
        print("✗ huggingface_hub not installed. Run: pip install huggingface-hub")
        vector_path = None
    except Exception as e:
        print(f"✗ Download failed: {e}")
        vector_path = None
else:
    print(f"✓ Vector file found: {vector_path}")

if vector_path and os.path.exists(vector_path):
    # Initialize with word embeddings
    m_vec = mecrab.MeCrab(vector_path=vector_path)
    print("✓ MeCrab initialized with word embeddings")
else:
    print("  See section 4 below for manual download instructions")
    m_vec = None

### Semantic Similarity Examples

In [ ]:
if m_vec is not None:
    # Compare similar words
    word_pairs = [
        ("東京", "京都"),      # Similar (cities)
        ("東京", "大阪"),      # Similar (cities)
        ("東京", "食べる"),    # Dissimilar
        ("猫", "犬"),          # Similar (animals)
        ("学校", "大学"),      # Similar (education)
    ]
    
    print("Semantic Similarity (Cosine)\n" + "="*50)
    for word1, word2 in word_pairs:
        try:
            sim = m_vec.similarity(word1, word2)
            print(f"{word1:8s} ⟷ {word2:8s} : {sim:6.3f}")
        except RuntimeError as e:
            print(f"{word1:8s} ⟷ {word2:8s} : Error - {e}")

### Combined: IPA + Embeddings

In [ ]:
if m_vec is not None:
    # Initialize with both IPA and vectors
    m_full = mecrab.MeCrab(with_ipa=True, vector_path=vector_path)
    
    text = "東京に行く"
    morphemes = m_full.parse_to_dict(text)
    
    print(f"Parsed: {text}\n")
    for morph in morphemes:
        print(f"Surface: {morph['surface']}")
        print(f"  IPA: /{morph.get('ipa', 'N/A')}/")
        
        if 'embedding' in morph:
            emb = morph['embedding']
            print(f"  Embedding: {len(emb)}-dim vector, first 5: {emb[:5]}")
        print()

## 4. Getting Word Vectors

### Option A: Download Pre-trained Vectors (Recommended)

**Japanese Wikipedia Word2Vec Vectors** - trained on full Wikipedia corpus:

**Dataset:** https://huggingface.co/datasets/KitaSan/mecrab-jawiki-word2vec

**Details:**
- Source: Japanese Wikipedia (全文)
- Vocabulary: 163,922 words (IPADIC)
- Vector Size: 100 dimensions
- Training: Hogwild! algorithm (83% parallel efficiency on 6 cores)
- Format: MCV1 (memory-mapped, instant loading)

**Download via CLI:**

In [ ]:
!pip install -q huggingface-hub
!huggingface-cli download KitaSan/mecrab-jawiki-word2vec vectors.bin --local-dir .

**Or download via Python API:**

In [ ]:
from huggingface_hub import hf_hub_download

vectors_path = hf_hub_download(
    repo_id="KitaSan/mecrab-jawiki-word2vec",
    filename="vectors.bin",
    repo_type="dataset"
)
print(f"Downloaded to: {vectors_path}")

### Option B: Train Custom Vectors

For domain-specific corpora, train your own vectors using KizaMe CLI:

```bash
# 1. Extract vocabulary
kizame dict dump -d /var/lib/mecab/dic/ipadic-utf8 --vocab > vocab.txt
MAX_WORD_ID=$(tail -1 vocab.txt | cut -f1)

# 2. Parse corpus to word_id sequences
cat corpus.txt | kizame parse --wakati-word-id > corpus_ids.txt

# 3. Train Word2Vec (Pure Rust, 83% parallel efficiency on 6 cores!)
kizame vectors train \
  -i corpus_ids.txt \
  -o vectors.bin \
  -f mcv1 \
  --max-word-id $MAX_WORD_ID \
  --size 100 \
  --window 5 \
  --negative 5 \
  --epochs 3 \
  --threads 6
```

See `../../kizame/WORD2VEC_TRAINING_GUIDE.md` for complete training pipeline.

## 5. Practical Application: Phonetic Search

Build a phonetic search engine using IPA.

In [ ]:
# Sample database of phrases
phrases_db = [
    "東京に行く",
    "京都で遊ぶ",
    "朝ごはんを食べる",
    "友達と会う",
    "本を読む"
]

# Index by IPA pronunciation
ipa_index = {}
for phrase in phrases_db:
    ipa = m_ipa.to_ipa_text(phrase)
    ipa_index[ipa] = phrase

# Display index
print("Phonetic Index:\n" + "="*70)
for ipa, phrase in sorted(ipa_index.items()):
    print(f"/{ipa:40s}/ → {phrase}")

### Search by Approximate Pronunciation

In [ ]:
# Search for phrases containing "kʲo" sound
query_sound = "kʲo"
results = [(ipa, phrase) for ipa, phrase in ipa_index.items() if query_sound in ipa]

print(f"Phrases containing /{query_sound}/:\n")
for ipa, phrase in results:
    print(f"  {phrase} → /{ipa}/")

## 6. Practical Application: Semantic Clustering

Group words by semantic similarity using embeddings.

In [ ]:
if m_vec is not None:
    # Sample vocabulary
    words = ["東京", "京都", "大阪", "猫", "犬", "学校", "大学", "食べる", "飲む"]
    
    # Compute pairwise similarities
    print("Pairwise Similarity Matrix:\n")
    print(" " * 10 + " ".join(f"{w:>6s}" for w in words[:5]))  # Show first 5
    print("="*60)
    
    for w1 in words[:5]:
        row = [f"{w1:8s}"]
        for w2 in words[:5]:
            try:
                sim = m_vec.similarity(w1, w2)
                row.append(f"{sim:6.3f}")
            except:
                row.append("  N/A")
        print(" ".join(row))

## Summary

### Key Features Demonstrated:

1. **IPA Pronunciation** (One-shot API)
   - `to_ipa(text)` → list of IPA strings
   - `to_ipa_text(text, separator=" ")` → joined IPA string

2. **Pythonic Dictionary API**
   - `parse_to_dict(text)` → list of dicts with all fields
   - Includes: `surface`, `pos`, `reading`, `ipa`, `embedding`

3. **Word Embeddings**
   - `similarity(word1, word2)` → cosine similarity
   - Automatic morpheme-to-embedding lookup
   - Pre-trained vectors from HuggingFace

### Use Cases:

- **Language Learning:** Phonetic pronunciation guides
- **Speech Synthesis:** IPA → speech engines
- **Semantic Search:** Find similar documents/words
- **Text Classification:** Use embeddings as features
- **Chatbots/NLP:** Understand intent via similarity

## Resources

- **Pre-trained Vectors:** https://huggingface.co/datasets/KitaSan/mecrab-jawiki-word2vec
- **Training Guide:** `../../kizame/WORD2VEC_TRAINING_GUIDE.md`
- **Python API Reference:** `../README.md`
- **GitHub:** https://github.com/kitasan/mecrab

## Next Steps

- Download pre-trained vectors from HuggingFace
- Train custom Word2Vec on your domain-specific corpus
- Build production NLP pipelines with MeCrab
- Integrate with ML models (scikit-learn, PyTorch, etc.)

---

**Copyright 2026 COOLJAPAN OU (Team KitaSan)**  
**License:** MIT OR Apache-2.0